# The Product Pricer Continued

A model that can estimate how much something costs, from its description.

## Enter The Frontier!

And now - we put Frontier Models to the test.

### 2 important points:

It's important to appreciate that we aren't Training the frontier models. We're only providing them with the Test dataset to see how they perform. They don't gain the benefit of the 400,000 training examples that we provided to the Traditional ML models.

HAVING SAID THAT...

It's entirely possible that in their monstrously large training data, they've already been exposed to all the products in the training AND the test set. So there could be test "contamination" here which gives them an unfair advantage. We should keep that in mind.

### import tất cả các thư viện cần thiết cho dự án:

- os: tương tác với hệ điều hành, quản lý đường dẫn và biến môi trường
- re: xử lý regular expressions, sẽ được dùng để trích xuất số từ chuỗi
- math và numpy: hỗ trợ tính toán
- json: xử lý dữ liệu định dạng JSON
- random: tạo số ngẫu nhiên
- dotenv: quản lý biến môi trường từ file .env
- huggingface_hub: kết nối với Hugging Face Hub
- matplotlib: tạo biểu đồ và trực quan hóa
- pickle: lưu và tải các đối tượng Python
- Counter: đếm phần tử trong collection
- OpenAI và Anthropic: các client API để giao tiếp với mô hình của các công ty tương ứng

In [ ]:
# imports

import os
import re
import math
import json
import random
from dotenv import load_dotenv
from huggingface_hub import login
import matplotlib.pyplot as plt
import numpy as np
import pickle
from collections import Counter
from openai import OpenAI
from anthropic import Anthropic

- Gọi load_dotenv(override=True) để tải các biến từ file .env vào môi trường, với override=True cho phép ghi đè các biến đã tồn tại
- Thiết lập ba biến môi trường quan trọng cho các API:

   - OPENAI_API_KEY: Key để sử dụng các mô hình của OpenAI (GPT-4o-mini, GPT-4o)
   - ANTHROPIC_API_KEY: Key để sử dụng mô hình của Anthropic (Claude 3.5 Sonnet)
   - HF_TOKEN: Token để đăng nhập vào Hugging Face Hub


- Mỗi biến được lấy từ biến môi trường hiện có, và nếu không tìm thấy sẽ sử dụng giá trị mặc định 'your-key-if-not-using-env'

In [ ]:
# environment

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

ĐĐăng nhập vào Hugging Face Hub bằng token đã thiết lập, và thêm vào git credential để có thể tải xuống các mô hình cần thiết.

In [ ]:
# Log in to HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

##  Import các module tùy chỉnh và khởi tạo client


Đoạn này import hai module tùy chỉnh:

- Item: Đại diện cho một mặt hàng với mô tả và giá cả
- Tester: Chứa các phương thức để kiểm tra hiệu suất của các mô hình dự đoán giá

In [ ]:
# moved our Tester into a separate package
# call it with Tester.test(function_name, test_dataset)

from items import Item
from testing import Tester

Khởi tạo các client API:

- openai: Để tương tác với API của OpenAI
- claude: Để tương tác với API của Anthropic

In [ ]:
openai = OpenAI()
claude = Anthropic()

Câu lệnh Jupyter magic này đảm bảo các biểu đồ matplotlib hiển thị ngay trong notebook.

In [ ]:
%matplotlib inline

## Tải dữ liệu


Đoạn này tải hai bộ dữ liệu từ các file pickle đã được lưu trước đó:

- train.pkl: Chứa khoảng 400,000 mẫu dữ liệu huấn luyện như đề cập trong phần giới thiệu
- test.pkl: Chứa dữ liệu kiểm tra để đánh giá hiệu suất các mô hình
- Các file pickle lưu các đối tượng Python nguyên vẹn, giúp tránh phải xử lý dữ liệu lại từ đầu

In [ ]:
# Let's avoid curating all our data again! Load in the pickle files:

with open('train.pkl', 'rb') as file:
    train = pickle.load(file)

with open('test.pkl', 'rb') as file:
    test = pickle.load(file)

# Before we look at the Frontier

## There is one more model we could consider

## Tạo file CSV cho dự đoán bởi con người

- Tạo một file CSV chứa 250 mẫu đầu tiên từ tập test
- Mỗi dòng chứa prompt kiểm tra của item và giá trị 0 (sẽ được thay thế bằng dự đoán của con người)
- Sau đó, con người sẽ đọc file này, điền giá dự đoán vào cột thứ hai, và lưu lại kết quả

In [ ]:
# Write the test set to a CSV

import csv
with open('human_input.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:250]:
        writer.writerow([t.test_prompt(), 0])

ĐĐọc lại file sau khi con người đã điền giá dự đoán và lưu vào list human_predictions.

In [ ]:
# Read it back in

human_predictions = []
with open('human_output.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

Hàm này trả về giá dự đoán của con người cho một item trong tập test, bằng cách tìm vị trí của item đó trong list test và lấy giá trị tương ứng từ human_predictions.

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
Tester.test(human_pricer, test)

## First, the humble but mighty GPT-4o-mini

It's called mini, but it packs a punch.

## Thiết lập các hàm để tương tác với mô hình AI

Hàm này chuẩn bị các thông điệp để gửi tới các mô hình AI:

- system_message: Hướng dẫn mô hình chỉ trả về giá, không giải thích
- user_prompt: Lấy prompt kiểm tra từ item, nhưng loại bỏ:

   - " to the nearest dollar" - vì frontier models không cần phải làm tròn số
   - "\n\nPrice is $" - phần này có thể là hậu tố trong prompt gốc


- Trả về một mảng các message theo định dạng chuẩn cho chat API:

   - Message hệ thống với nội dung là system_message
   - Message người dùng với nội dung là user_prompt
   - Message trợ lý (để gợi ý format) với nội dung là "Price is $"

In [ ]:
# First let's work on a good prompt for a Frontier model
# Notice that I'm removing the " to the nearest dollar"
# When we train our own models, we'll need to make the problem as easy as possible, 
# but a Frontier model needs no such simplification.

def messages_for(item):
    system_message = "You estimate prices of items. Reply only with the price, no explanation"
    user_prompt = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": "Price is $"}
    ]

In [ ]:
# Try this out

messages_for(test[0])

Hàm này xử lý kết quả trả về từ các mô hình AI để lấy giá trị số:

- Loại bỏ ký tự '$' và dấu phẩy trong số
- Sử dụng regular expression để tìm số thập phân hoặc số nguyên đầu tiên trong chuỗi

   - [-+]?: Dấu cộng hoặc trừ tùy chọn
   - \d*\.\d+: Số thập phân (phần nguyên có thể có hoặc không, phần thập phân bắt buộc)
   - \d+: Số nguyên


- Trả về giá trị số dạng float nếu tìm thấy, nếu không trả về 0

In [ ]:
# A utility function to extract the price from a string

def get_price(s):
    s = s.replace('$','').replace(',','')
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else 0

In [ ]:
get_price("The price is roughly $99.99 because blah blah")

## Các hàm gọi API của các mô hình frontier

Hàm này gọi API của OpenAI để dự đoán giá bằng mô hình GPT-4o-mini:

- model="gpt-4o-mini": Xác định mô hình cần sử dụng
- messages=messages_for(item): Sử dụng hàm messages_for() để tạo prompt
- seed=42: Đặt seed để đảm bảo tính nhất quán giữa các lần gọi
- max_tokens=5: Giới hạn đầu ra không quá 5 token (đủ cho một số giá)
- Sau đó trích xuất nội dung từ phản hồi và chuyển đổi thành số bằng get_price()

In [ ]:
# The function for gpt-4o-mini

def gpt_4o_mini(item):
    response = openai.chat.completions.create(
        model="gpt-4o-mini", 
        messages=messages_for(item),
        seed=42,
        max_tokens=5
    )
    reply = response.choices[0].message.content
    return get_price(reply)

In [ ]:
test[0].price

In [ ]:
Tester.test(gpt_4o_mini, test)

Hàm này sử dụng mô hình GPT-4o phiên bản tháng 8/2024, được coi là mô hình tiên tiến hơn so với GPT-4o-mini.

In [ ]:
def gpt_4o_frontier(item):
    response = openai.chat.completions.create(
        model="gpt-4o-2024-08-06", 
        messages=messages_for(item),
        seed=42,
        max_tokens=5
    )
    reply = response.choices[0].message.content
    return get_price(reply)

In [ ]:
# The function for gpt-4o - the August model
# Note that it cost me about 1-2 cents to run this (pricing may vary by region)
# You can skip this and look at my results instead

Tester.test(gpt_4o_frontier, test)

### Hàm này gọi API của Anthropic để dự đoán giá bằng mô hình Claude 3.5 Sonnet:

- Đầu tiên, tách system message ra khỏi các message khác vì API của Anthropic có cấu trúc khác với OpenAI
- model="claude-3-5-sonnet-20240620": Xác định mô hình Claude 3.5 Sonnet phiên bản 20/06/2024
- max_tokens=5: Giới hạn đầu ra giống như với các mô hình GPT
- system=system_message: Truyền system message riêng
- messages=messages: Truyền các message còn lại
- Trích xuất nội dung văn bản từ phản hồi và chuyển đổi thành số

In [ ]:
def claude_3_point_5_sonnet(item):
    messages = messages_for(item)
    system_message = messages[0]['content']
    messages = messages[1:]
    response = claude.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=5,
        system=system_message,
        messages=messages
    )
    reply = response.content[0].text
    return get_price(reply)

### Các dòng này gọi phương thức test() từ module Tester để đánh giá hiệu suất của mỗi phương pháp dự đoán giá:

- human_pricer: Dự đoán từ con người (chỉ có cho 250 mẫu đầu tiên)
- gpt_4o_mini: Mô hình nhỏ của OpenAI
- gpt_4o_frontier: Mô hình tiên tiến hơn của OpenAI
- claude_3_point_5_sonnet: Mô hình của Anthropic

Phương thức test() sẽ chạy mỗi hàm này với tất cả các mẫu trong tập test, so sánh với giá thực tế, và tính toán các chỉ số hiệu suất như sai số trung bình, độ chính xác, v.v.

In [ ]:
# The function for Claude 3.5 Sonnet
# It also cost me about 1-2 cents to run this (pricing may vary by region)
# You can skip this and look at my results instead

Tester.test(human_pricer, test)
Tester.test(gpt_4o_mini, test)
Tester.test(gpt_4o_frontier, test)
Tester.test(claude_3_point_5_sonnet, test)